In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/datasets/spe-1/spe1_helper_modules/')
from spk_lfp_cluster_comp_analysis import *
from spk_feat_cluster_comp_analysis import compile_experiment_results

In [ ]:
# Load LFP sliding window stats
df_pop_stats, pop_traces = compile_lfp_stats()

In [ ]:
# Load cluster + metadata master table
cluster_pickle_dir = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spe1_pickles/cluster_pickles/'
df_master = compile_experiment_results(cluster_pickle_dir)

#### Merging LFP stats with cluster metadata

We join the LFP sliding-window results (from `lfp_spk_cluster_comparisons_target_cells`) with the spike cluster metadata (from `spk_waveform_cluster_comparisons`) on `cell_id` × `spike_feature`. The result is a flat DataFrame where every row is one (cell, spike feature, LFP feature) triplet, carrying both the LFP effect size / yield and the clustering quality metrics for that cell × feature pair. This lets us ask: do cells that cluster more cleanly also couple more strongly to the LFP?

In [ ]:
# Normalize spike_feature names for joining
# df_pop_stats has e.g. 'peak_amp_cluster', df_master has 'peak_amp'
df_pop_stats['spike_feature_base'] = df_pop_stats['spike_feature'].str.replace('_cluster', '')

# Aggregate df_master to one row per cell x spike_feature (drop cluster-level rows)
meta_cols = ['cell_id', 'spike_feature', 'patch_type', 'current_type', 'cell_type',
             'cortical_depth', 'dark_neuron', 'clear_EAP_waveform',
             'num_clusters', 'nRMSE', 'cos_sim', 'temporal_rho', 'temporal_p', 'temporal_component']

df_meta = (
    df_master[meta_cols]
    .copy()
    .assign(**{c: pd.to_numeric(df_master[c], errors='coerce')
               for c in ['nRMSE', 'cos_sim', 'temporal_rho', 'temporal_component', 'cortical_depth']})
    .groupby(['cell_id', 'spike_feature'], as_index=False)
    .agg({
        'patch_type': 'first', 'current_type': 'first', 'cell_type': 'first',
        'cortical_depth': 'first', 'dark_neuron': 'first', 'clear_EAP_waveform': 'first',
        'num_clusters': 'first',
        'nRMSE': 'mean', 'cos_sim': 'mean',
        'temporal_rho': 'first', 'temporal_p': 'first', 'temporal_component': 'first'
    })
)

# Aggregate LFP stats to one row per cell x spike_feature x lfp_feature
df_lfp_agg = (
    df_pop_stats
    .groupby(['cell_id', 'spike_feature_base', 'lfp_feature'], as_index=False)
    .agg(
        n_sig_windows=('cohens_d', 'count'),
        median_cohens_d=('cohens_d', 'median'),
        max_cohens_d=('cohens_d', 'max'),
        min_pvalue=('p_value', 'min'),
    )
    .rename(columns={'spike_feature_base': 'spike_feature'})
)

# Merge
df_merged = df_lfp_agg.merge(df_meta, on=['cell_id', 'spike_feature'], how='left')
print(f'Merged df: {df_merged.shape}')
df_merged.head()

#### Do cells with bigger waveform differences show stronger LFP effects?

If spike waveform clustering reflects real modulation by network state, then cells with more separated clusters (high nRMSE, low cos_sim) should also show larger LFP effect sizes — more distinct waveform shapes should coincide with more distinct LFP environments. We test this with Spearman correlations between median Cohen's d and each clustering quality metric, separately for each LFP feature.

In [ ]:
from scipy.stats import spearmanr

clust_metrics = ['nRMSE', 'cos_sim', 'temporal_rho']
lfp_feats = sorted(df_merged['lfp_feature'].unique())

fig, axes = plt.subplots(len(lfp_feats), len(clust_metrics),
                          figsize=(5 * len(clust_metrics), 4 * len(lfp_feats)))

for i, lfp in enumerate(lfp_feats):
    for j, met in enumerate(clust_metrics):
        ax = axes[i, j]
        sub = df_merged[df_merged['lfp_feature'] == lfp][['median_cohens_d', met]].dropna()
        ax.scatter(sub[met], sub['median_cohens_d'], alpha=0.5, s=30, color='steelblue')
        if len(sub) >= 3:
            rho, p = spearmanr(sub[met], sub['median_cohens_d'])
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            p_str = f'{p:.4f}' if p >= 0.0001 else '<0.0001'
            ax.set_title(f'{lfp}\nvs {met}\nSpearman r={rho:.2f}, {sig} (p={p_str})', fontsize=9, fontweight='bold')
        else:
            ax.set_title(f'{lfp} vs {met}\n(n<3)', fontsize=9)
        ax.set_xlabel(met, fontsize=9)
        ax.set_ylabel("Median Cohen's d", fontsize=9)
        sns.despine(ax=ax)

plt.suptitle('LFP Effect Size vs Clustering Quality Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### Does cell metadata predict LFP effect strength?

We ask whether recording variables — cell type, patch type, current type, dark neuron flag, EAP quality, and cortical depth — are associated with how strongly or how often LFP effects appear. Kruskal-Wallis for categorical variables; Spearman for cortical depth. A significant association means LFP-spike coupling is not uniform across the population, which could point toward a cell-type or layer-specific mechanism worth following up on.

In [ ]:
from scipy.stats import kruskal

cat_meta = ['cell_type', 'patch_type', 'current_type', 'dark_neuron', 'clear_EAP_waveform']
cont_meta = ['cortical_depth']

for lfp in lfp_feats:
    sub = df_merged[df_merged['lfp_feature'] == lfp].copy()
    n_plots = len(cat_meta) + len(cont_meta)
    fig, axes = plt.subplots(2, (n_plots + 1) // 2, figsize=(16, 10))
    axes = axes.flatten()

    for ax_idx, col in enumerate(cat_meta + cont_meta):
        ax = axes[ax_idx]
        plot_sub = sub[['median_cohens_d', col]].dropna()
        if col in cont_meta:
            x = pd.to_numeric(plot_sub[col], errors='coerce')
            y = plot_sub['median_cohens_d']
            valid = np.isfinite(x) & np.isfinite(y)
            ax.scatter(x[valid], y[valid], alpha=0.5, s=30, color='steelblue')
            if valid.sum() >= 3:
                rho, p = spearmanr(x[valid], y[valid])
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                p_str = f'{p:.4f}' if p >= 0.0001 else '<0.0001'
                ax.set_title(f'{col}\nSpearman r={rho:.2f}, {sig} (p={p_str})', fontsize=9, fontweight='bold')
            ax.set_xlabel(col, fontsize=9)
            ax.set_ylabel("Median Cohen's d", fontsize=9)
        else:
            groups = [g['median_cohens_d'].values for _, g in plot_sub.groupby(col) if len(g) > 0]
            stat, p = kruskal(*groups) if len(groups) >= 2 else (np.nan, np.nan)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            p_str = f'{p:.4f}' if pd.notnull(p) and p >= 0.0001 else ('<0.0001' if pd.notnull(p) else 'n/a')
            sns.boxplot(data=plot_sub, x=col, y='median_cohens_d', showfliers=False, palette='Paired', ax=ax)
            sns.stripplot(data=plot_sub, x=col, y='median_cohens_d', color='.3', alpha=0.4, ax=ax)
            ax.set_title(f'{col}\nKW {sig} (p={p_str})', fontsize=9, fontweight='bold')
            ax.set_xlabel(col, fontsize=9)
            ax.set_ylabel("Median Cohen's d", fontsize=9)
        sns.despine(ax=ax)

    for j in range(ax_idx + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f'LFP Feature: {lfp}\nMetadata vs Median Effect Size', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

#### Does temporal drift predict LFP effect strength?

Cells with significant temporal drift (`temporal_component = 1`) are borderline cases — their LFP effects could reflect slow recording changes rather than fast network modulation of spike waveforms. Here we directly test whether drift status predicts either the number of significant LFP windows or the median Cohen's d. If drifting cells consistently show stronger LFP effects, that's a red flag: it suggests at least some of our effects are confounded by non-stationarity rather than genuine coupling.

In [ ]:
from scipy.stats import kruskal

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric in zip(axes, ['n_sig_windows', 'median_cohens_d']):
    sub = df_merged[['temporal_component', metric]].dropna()
    sub['temporal_component'] = sub['temporal_component'].astype(float).map({0.0: 'No drift', 1.0: 'Sig drift'})
    order = ['No drift', 'Sig drift']
    sns.boxplot(data=sub, x='temporal_component', y=metric, order=order,
                palette=['lightblue', 'salmon'], showfliers=False, ax=ax)
    sns.stripplot(data=sub, x='temporal_component', y=metric, order=order,
                  color='.3', alpha=0.4, ax=ax)
    groups = [g[metric].values for _, g in sub.groupby('temporal_component') if len(g) > 0]
    stat, p = kruskal(*groups) if len(groups) >= 2 else (np.nan, np.nan)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    p_str = f'{p:.4f}' if pd.notnull(p) and p >= 0.0001 else '<0.0001'
    ax.set_title(f'Temporal component vs {metric}\n{sig} (p={p_str})', fontweight='bold')
    sns.despine(ax=ax)

plt.suptitle('Does Temporal Drift Predict LFP Effect Strength?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Which cell × spike feature × LFP feature triplets are the most compelling?

We filter to triplets that have (a) at least one significant LFP window and (b) above-median clustering quality (high nRMSE or low cos_sim) — cases where both the spike-level and LFP-level evidence are strong. Ranked by median Cohen's d, this table flags the clearest candidates for further single-cell or population-level follow-up.

In [ ]:
nRMSE_thresh = df_merged['nRMSE'].quantile(0.5)
cos_sim_thresh = df_merged['cos_sim'].quantile(0.5)

df_highlight = df_merged[
    (df_merged['n_sig_windows'] > 0) &
    ((df_merged['nRMSE'] >= nRMSE_thresh) | (df_merged['cos_sim'] <= cos_sim_thresh))
].sort_values('median_cohens_d', ascending=False)

cols_show = ['cell_id', 'spike_feature', 'lfp_feature', 'n_sig_windows', 'median_cohens_d',
             'nRMSE', 'cos_sim', 'temporal_rho', 'temporal_component',
             'cell_type', 'cortical_depth']
df_highlight[cols_show].reset_index(drop=True)